In [ ]:
# 2_removing_artifacts.ipynb
# Cell tag parameters (for batch processing — papermill reads these and loops through the participants)
# PAPERMILL_RUN is toggled "True" if batch processing is being used
p_id = 'participant_1'
PAPERMILL_RUN = False

In [ ]:
# Imports
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from autoreject import AutoReject

# Allow imports to work regardless of the notebook's location by locating the project root.
# EEG_DIR is defined in config.py
def find_project_root(current_path, marker='config.py'):
    path = Path(current_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path 

root_dir = find_project_root(Path.cwd())
sys.path.append(str(root_dir))

from config import EEG_DIR, BEHAV_DIR, TRIGGER_MAP

mne.set_log_level('error') # reduce extraneous MNE output

# Plotting Toggle: it is deactivated if running each notebook manually, but if batch processing
# is being used, it is activated so windows don't pop-up
if PAPERMILL_RUN:
    # Batch Mode: Suppress pop-ups and draw plots inline
    %matplotlib inline
    mne.viz.set_browser_backend('matplotlib')
    show_plots = False
else:
     # Manual Mode: Allow interactive pop-ups for data exploration
    %matplotlib auto
    show_plots = True

In [ ]:
# Path management
subj_dir = EEG_DIR / p_id # each participant's folder
csv_path = BEHAV_DIR / 'full_merged_data.csv'

In [ ]:
# Load EEG file
raw = mne.io.read_raw_fif(subj_dir / f'{p_id}-raw.fif', preload=True)
events, _ = mne.events_from_annotations(raw)
eeg_codes = events[:, 2]

# Load E-Prime file
df = pd.read_csv(csv_path, sep=',', encoding='utf-8', low_memory=False)
subject_num = int(p_id.split('_')[-1]) 
df = df[df['Subject'] == subject_num].reset_index(drop=True)

# Filter strictly by runtime duration (removes phantom E-Prime rows)
df = df.dropna(subset=['Dur_Word_1']).reset_index(drop=True)

print(f"E-Prime and EEG (raw) files loaded: {len(df)} E-Prime Trials for {len(eeg_codes)} EEG events")

In [ ]:
# Plot the file before labeling it
raw.plot(events=events, duration=10, n_channels=30, scalings=dict(eeg=50e-6), title="BEFORE Synchronization", show=show_plots);

In [ ]:
# 1. Synchronization: This aligns E-prime and EEG data, so we can prepare for ICA and Autoreject 
# (it allows us to find and remove training blocks, pauses, etc.)
# The training was a Condition B sentence and it inadvertently sent the Condition B trigger. This is treated systematically
# in the pipeline.
#
# TRIGGER_MAP is imported from config.py. It is the correspondence between E-Prime and EEG triggers

# Prepare Arrays
eeg_codes = eeg_codes.astype(int)

# Apply mapping: Now csv_codes will contain 1004, 1010, etc. instead of 64, 160...
csv_codes = df['TrigCode'].map(TRIGGER_MAP).fillna(0).astype(int).to_numpy()

# Sliding Window
# Now it checks for matching patterns using the translated codes
matches = [np.sum(eeg_codes == csv_codes[i:i+len(eeg_codes)]) 
           for i in range(len(csv_codes) - len(eeg_codes) + 1)]

if not matches:
    raise ValueError(f"CRITICAL ERROR: EEG pattern not found in CSV for {p_id}.")

# Find alignment
offset = int(np.argmax(matches))
match_score = matches[offset]
match_pct = match_score / len(eeg_codes) * 100

# Raises an error if matching is below 99%
if match_pct < 99.0:
    raise ValueError(
        f"{p_id}: weak sync ({match_pct:.1f}%, {match_score}/{len(eeg_codes)}) — investigate."
    )


print(f"✅ Synced at CSV Row {offset}")
print(f"Match Quality: {match_score}/{len(eeg_codes)} triggers matched ({(match_score/len(eeg_codes))*100:.1f}%)")

# 2. Compute trial timing and create annotations

onsets, durations, descs = [], [], []

def get_relative_times(row, trig_time):
    words = [row[f'Word_{j}'] for j in range(1, 8) if pd.notna(row[f'Word_{j}'])]
    try:
        t_idx = next(x for x, w in enumerate(words) if '.png' in str(w))
    except StopIteration: return None, None, None

    t_start = trig_time - (1.5 + (t_idx * 0.5))
    t_end_sent = trig_time + (1.0 + ((len(words) - 1 - t_idx) * 0.5))
    t_end_jitter = t_end_sent + (row['jitteredinterval'] / 1000.0)
    return t_start, t_end_sent, t_end_jitter

for i, eeg_code in enumerate(eeg_codes):
    
    # Use the calculated offset
    if (idx := i + offset) >= len(df): break
    
    row = df.iloc[idx]
    trig_time = events[i, 0] / raw.info['sfreq']
    
    # Calculate timings
    t_start, t_sent_end, t_jitter_end = get_relative_times(row, trig_time)
    if t_start is None: continue

    # Labeling
    if str(row['Running']) == 'Training':
        label = "BAD_Training"
    else:
        label = f"Sentence_{row['Condition']}"

    # Add the annotation for the current trial
    onsets.append(t_start)
    durations.append(t_jitter_end - t_start)
    descs.append(label)

    # Annotate the interval between consecutive trials
    if i < len(eeg_codes) - 1:
        if idx + 1 < len(df):
            next_row = df.iloc[idx + 1]
            next_trig_time = events[i+1, 0] / raw.info['sfreq']
            t_next_start, _, _ = get_relative_times(next_row, next_trig_time)
            
            if t_next_start and t_next_start > t_jitter_end:
                onsets.append(t_jitter_end)
                durations.append(t_next_start - t_jitter_end)
                descs.append("BAD_Gap")
    else:
        # Annotate the recording after the final trial
        file_end_time = raw.times[-1]
        if file_end_time > t_jitter_end:
            onsets.append(t_jitter_end)
            durations.append(file_end_time - t_jitter_end)
            descs.append("BAD_End")

# Annotate the recording period before the first trial (setup)
if onsets and onsets[0] > 0:
    onsets.append(0)
    durations.append(onsets[0])
    descs.append("BAD_Setup")

# Apply the annotations to the raw recording
raw.set_annotations(mne.Annotations(onsets, durations, descs))

# Verify that annotations do not overlap
check_data = sorted(zip(onsets, durations, descs))
n_overlaps = sum(1 for i in range(len(check_data)-1) 
                 if check_data[i][0] + check_data[i][1] > check_data[i+1][0] + 0.001)

if n_overlaps == 0:
    print("✅ Success: Annotations applied with 0 overlaps.")
else:
    print(f"⚠️ Warning: {n_overlaps} trials overlapped.")

In [ ]:
# Verify alignment: Look for colored blocks around the triggers
raw.plot(events=events, duration=10, n_channels=30, scalings=dict(eeg=50e-6), title="AFTER Synchronization", show=show_plots);

In [ ]:
# Filter settings (different from those used in 1_filtering.ipynb)
ica_low_cut = 1.0 # For ICA, we filter out more low-frequency power
hi_cut  = 30

raw_ica = raw.copy().filter(ica_low_cut, hi_cut)

In [ ]:
# After applying the filters for ICA
raw_ica.plot(events=events, duration=10, n_channels=30, scalings=dict(eeg=50e-6), title="ICA plot", show=show_plots);

In [ ]:
# Create temporary 1 s epochs for AutoReject and ICA
tstep = 1.0
events_ica = mne.make_fixed_length_events(raw_ica, duration=tstep)
epochs_ica = mne.Epochs(raw_ica, events_ica,
                        tmin=0.0, tmax=tstep,
                        baseline=None,
                        preload=True)

In [ ]:
# Run Autoreject on the data
ar = AutoReject(n_interpolate=[1, 2, 4],
                random_state=42,
                picks=mne.pick_types(epochs_ica.info, 
                                     eeg=True,
                                     eog=False
                                    ),
                n_jobs=1, 
                verbose=False
                )

ar.fit(epochs_ica)

reject_log = ar.get_reject_log(epochs_ica)

In [ ]:
# Plot the results of AutoReject
fig, ax = plt.subplots(figsize=[15, 5])
reject_log.plot('horizontal', ax=ax, aspect='auto', show=show_plots)
plt.show()

In [ ]:
# ICA parameters
random_state = 42   # ensures ICA is reproducible each time it's run
ica_n_components = .99     # ICA that explains 99% of the variance

# Fit ICA
ica = mne.preprocessing.ICA(n_components=ica_n_components,
                            random_state=random_state,
                            )
ica.fit(epochs_ica[~reject_log.bad_epochs], 
decim=3)

In [ ]:
# Inspect ICA components
ica.plot_components(show=show_plots);

In [ ]:
# Inspect ICA properties
ica.plot_properties(epochs_ica, picks=range(0, ica.n_components_), psd_args={'fmax': hi_cut}, show=show_plots);

In [ ]:
# Adjust the z-score threshold until up to max_ic EOG components are identified

ica.exclude = []
num_excl = 0
max_ic = 2
z_thresh = 3.5
z_step = .05

while num_excl < max_ic:
    eog_indices, eog_scores = ica.find_bads_eog(epochs_ica,
                                                ch_name=['Fp1', 'Fp2', 'F7', 'F8'], 
                                                threshold=z_thresh
                                                )
    num_excl = len(eog_indices)
    z_thresh -= z_step # won't impact things if num_excl is ≥ n_max_eog 

# assign the bad EOG components to the ICA.exclude attribute so they can be removed later
ica.exclude = eog_indices

print('Final z threshold = ' + str(round(z_thresh, 2)))

In [ ]:
# Inspect EOG correlation scores
ica.plot_scores(eog_scores, show=show_plots);

In [ ]:
# Save ICA
ica.save(subj_dir / f'{p_id}-ica.fif',  overwrite=True);

# Save annotations for later preprocessing steps
raw.annotations.save(subj_dir / f'{p_id}-annot.fif', overwrite=True);
print(f"✅ Saved ICA and Annotations for {p_id}")

In [ ]:
print(f"✅ Finished Removing Artifacts for {p_id}")